In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path("..").resolve()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.io import load_dataset, load_fine


DATA_DIR = REPO_ROOT / "DATA"
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

fine = load_fine(
    DATA_DIR,
    "fine.csv",
)

print(f"Repository root : {REPO_ROOT}")
print(f"Data directory  : {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
aggregated_mid = load_dataset(
    data_dir = DATA_DIR,
    filename = "aggregated_mid.csv",
    index_col=0
)

aggregation_dictionary = pd.read_csv(
    DATA_DIR / "dictionary_for_aggregated.csv",
    sep=";"
)


allen_coarse_colors = {
    "Isocortex": "#70ff71",
    "OLF": "#9ad2bd",
    "HPF": "#7ed04b",
    "CTXsp": "#8ada87",
    "STR": "#98d6f9",
    "PAL": "#8599cc",
    "TH": "#ff7080",
    "HY": "#e64438",
    "MB": "#ff64ff",
    "P": "#ff9b88",
    "MY": "#ff9bcd"
}


conditions = ['CNTX','OCT']

condition_datasets = {
    condition: aggregated_mid.filter(
        regex=rf"{condition}",
        axis=1
    )
    for condition in conditions
}

In [ ]:
import umap

plt.rcParams["font.family"] = "Arial"

palette = {
    "CNTX": "#66c2a5",
    "OCT": "#fc8d62",
    "OPCRT": "#8da0cb"
}

corr_matrices = {}

for condition_name, condition_data in condition_datasets.items():

    correlation_matrix = condition_data.T.corr()
    corr_matrices[condition_name] = correlation_matrix

    fig, ax = plt.subplots(figsize=(12, 9))

    heatmap = sns.heatmap(
        correlation_matrix,
        cmap="crest",
        vmin=-1,
        vmax=1,
        xticklabels=True,
        yticklabels=True,
        ax=ax
    )

    ax.tick_params(axis="x", rotation=90, labelsize=10)
    ax.tick_params(axis="y", labelsize=10)

    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_visible(False)

    ax.tick_params(
        left=False,
        bottom=False
    )

    colorbar = heatmap.collections[0].colorbar
    colorbar.ax.tick_params(labelsize=17)
    colorbar.outline.set_linewidth(2)

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title("")

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / f"{condition_name}_correlation_heatmap.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
from utils.network import compute_adj_matrix

adj_matrices = {}

for name, df in condition_datasets.items():
    adj_matrices[name] = compute_adj_matrix(df)

In [ ]:
from utils.network import build_brain_graph

brain_graphs = {}

for name, matrix in adj_matrices.items():
    results = build_brain_graph(
        adj_matrix=matrix,
        aggregated_codex=aggregation_dictionary,
        allen_coarse_colors=allen_coarse_colors,
        min_dist=0.5,
        iterations=60,
        seed=42,
        super_scale=5,
        group_scale=1,
        node_size=200,
        SAVE=True
    )
    brain_graphs[name] = results

In [ ]:
df_roles_cntx = pd.read_csv(DATA_DIR/'hub_cntx.csv', sep=';', index_col=0)
df_roles_oct = pd.read_csv(DATA_DIR/'hub_oct.csv', sep=';', index_col=0)

In [ ]:
from adjustText import adjust_text
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Arial"

conditions = {
    "CNTX": df_roles_cntx,
    "OCT": df_roles_oct,
}

role_list = [
    "R1: Ultra-peripheral",
    "R2: Peripheral",
    "R3: Non-hub connector",
    "R4: Non-hub kinless",
    "R5: Provincial hub",
    "R6: Connector hub",
    "R7: Kinless hub"
]

dot_colors = [
    "#bbd6e8",
    "#c1e2bf",
    "#f6b9ba",
    "#fff2b2",
    "#d2c4e0",
    "#ffd8b2",
    "#e4ccbe"
]

label_colors = [
    "#1F78B4",
    "#33A02C",
    "#E31A1C",
    "#FFD700",
    "#6A3D9A",
    "#FF7F00",
    "#A65628"
]

for condition_name, df_roles in conditions.items():

    plt.figure(figsize=(12, 6))

    all_texts = []

    for role, dot_color, label_color in zip(
        role_list, dot_colors, label_colors
    ):

        idx = df_roles["Role"] == role

        x = df_roles.loc[idx, "Participation_Coef"].values
        y = df_roles.loc[idx, "WithinModule_Z"].values
        nodes_in_role = df_roles.loc[idx, "Node"].values

        plt.scatter(
            x,
            y,
            label=role,
            s=400,
            color=dot_color
        )

        for xi, yi, node in zip(x, y, nodes_in_role):

            t = plt.text(
                xi,
                yi,
                str(node),
                fontsize=10,
                fontweight="bold",
                color=label_color
            )

            all_texts.append(t)

    adjust_text(
        all_texts,
        only_move={"points": "y", "texts": "xy"},
        autoalign="xy",
        expand_points=(1.2, 1.4),
        expand_text=(1.2, 1.4),
        force_points=0.05,
        force_text=0.05,
        arrowprops=dict(
            arrowstyle="-",
            color="none"
        )
    )

    plt.xlabel(
        "Participation Coefficient (P)",
        fontsize=14
    )

    plt.ylabel(
        "Within-Module Degree Z-score (z)",
        fontsize=14
    )

    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    plt.legend(
        bbox_to_anchor=(1.01, 1),
        loc="upper left",
        fontsize=12,
        labelspacing=1.2
    )

    plt.grid(False)

    plt.title(
        condition_name,
        fontsize=16,
        fontweight="bold"
    )

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / f"hub_roles_{condition_name.lower()}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()